# Chấm điểm động tác chống đẩy — Bước 6: thêm dữ liệu QEVD

**Vì sao cần bộ này.** Bộ Mendeley chỉ có 24 người và nhãn good/bad gắn cho cả video, nên mô hình dừng ở khoảng 66–68%.
QEVD-FIT-300K (Qualcomm) có hàng trăm người tập chống đẩy, mỗi clip khoảng 5 giây được gán nhãn chi tiết:
độ sâu (`shallow`, `90 degrees`, `deep`), hông (`elevated hips`, `butt sinking`), chân chạm sàn (`legs on the floor`),
vị trí tay (`wide`, `narrow`, ...).

**Đặc điểm.** Tất cả quay bằng webcam đặt trên sàn phía trước người tập, tức chỉ có **góc trước**.

**Đầu vào của notebook này** là keypoint đã trích sẵn (dataset `qevd_pushup_keypoints`), không phải video.
Video gốc nặng khoảng 22 GB mỗi Part nên được xử lý trên máy cá nhân bằng các script trong thư mục `qevd_tools/`:
1. `rut_clip_pushup.py`: rút riêng clip chống đẩy khỏi file nén chia phần.
2. `trich_keypoint.py`: chạy MediaPipe Pose (cùng mô hình với notebook 01), lấy 1 trong 2 khung hình để về 15 fps như Mendeley.
3. `dong_goi_kaggle.py`: đóng gói keypoint và nhãn thành file zip.

QEVD chỉ được dùng cho nghiên cứu, nên dataset trên Kaggle phải để **Private**.

**Đầu ra:** `qevd_dataset.npz`, cùng định dạng với `pushup_dataset.npz` của bước 3.

## 1. Tìm dữ liệu đầu vào

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd

found = glob.glob("/kaggle/working/**/qevd_pushup_clips.json", recursive=True) \
      + glob.glob("/kaggle/input/**/qevd_pushup_clips.json", recursive=True)
assert found, "Không thấy qevd_pushup_clips.json. Hãy Add Input dataset qevd_pushup_keypoints."

KP_ROOT = os.path.dirname(found[0])           # chứa qevd_pushup_clips.json và thư mục keypoints/
OUT_DIR = "/kaggle/working/qevd_dataset"
os.makedirs(OUT_DIR, exist_ok=True)

clips = json.load(open(found[0], encoding="utf-8"))
print("Số clip chống đẩy:", len(clips), "| số người:", len({c["worker_id"] for c in clips}))
print("Ví dụ:", clips[0])

## 2. Hàm đếm rep (giống hệt notebook 02)

Clip QEVD là góc trước nên dùng tín hiệu "độ cao vai so với cổ tay, chia cho độ rộng vai".

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, savgol_filter

VIS_THR = 0.5
MIN_AMP = {"front": 0.10, "side": 3.0, "diagonal": 3.0}     # biên độ tối thiểu của một rep, theo đơn vị của tín hiệu
L_SH, R_SH, L_EL, R_EL, L_WR, R_WR = 11, 12, 13, 14, 15, 16
L_HIP, L_ANK = 23, 27


def joint_angle(a, b, c):
    ba, bc = a - b, c - b
    cos = (ba * bc).sum(-1) / (np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def fill_gaps(x, valid):
    # Nội suy tuyến tính các khung không hợp lệ
    x = x.astype(np.float64).copy()
    idx = np.arange(len(x))
    if valid.sum() >= 2:
        x[~valid] = np.interp(idx[~valid], idx[valid], x[valid])
    return x


def near_side(vis):
    # 0 = bên trái, 1 = bên phải: bên nào tay nhìn rõ hơn
    return 0 if vis[:, [L_SH, L_EL, L_WR]].mean() >= vis[:, [R_SH, R_EL, R_WR]].mean() else 1


def rep_signal(d, view):
    kp = d["kp2d"]
    px = kp[:, :, :2] * np.array([int(d["width"]), int(d["height"])])   # về pixel để góc không bị méo
    vis, det = kp[:, :, 3], d["detected"]

    def in_frame(j):
        return (kp[:, j, 0] > -0.1) & (kp[:, j, 0] < 1.1) & (kp[:, j, 1] > -0.1) & (kp[:, j, 1] < 1.1)

    if view == "front":
        sh_y = px[:, [L_SH, R_SH], 1].mean(1)
        wr_y = px[:, [L_WR, R_WR], 1].mean(1)
        sh_w = np.linalg.norm(px[:, L_SH] - px[:, R_SH], axis=-1)
        valid = det & (vis[:, [L_SH, R_SH]].min(1) > VIS_THR) & (vis[:, [L_WR, R_WR]].max(1) > VIS_THR) \
            & in_frame(L_SH) & in_frame(R_SH)
        scale = np.median(sh_w[valid]) if valid.any() else 1.0
        sig = (wr_y - sh_y) / (scale + 1e-6)
    else:
        side = near_side(vis)
        s, e, w = L_SH + side, L_EL + side, L_WR + side
        sig = joint_angle(px[:, s], px[:, e], px[:, w])
        valid = det & (vis[:, [s, e, w]].min(1) > VIS_THR) & in_frame(s) & in_frame(e) & in_frame(w)

    sig = fill_gaps(sig, valid)
    if len(sig) >= 7:
        sig = savgol_filter(sig, 7, 2)      # làm mượt nhẹ
    return sig, valid


def find_reps(sig, valid, fps, min_amp=0.0, min_prom=0.35, min_rep_s=0.8, max_rep_s=8.0, min_valid=0.7):
    if valid.mean() < 0.3:      # video gần như không nhìn rõ tay: bỏ cả video
        return []
    lo, hi = np.percentile(sig[valid], [5, 95])
    rng = hi - lo
    bottoms, _ = find_peaks(-sig, prominence=min_prom * rng, distance=max(1, int(min_rep_s * fps * 0.6)))
    half = int(max_rep_s * fps / 2)
    reps = []
    for i, b in enumerate(bottoms):
        left = bottoms[i - 1] if i > 0 else max(0, b - half)
        right = bottoms[i + 1] if i + 1 < len(bottoms) else min(len(sig) - 1, b + half)
        start = left + int(np.argmax(sig[left:b + 1]))
        end = b + int(np.argmax(sig[b:right + 1]))
        dur = (end - start) / fps
        amp = float(min(sig[start], sig[end]) - sig[b])       # tín hiệu hạ xuống bao nhiêu so với đỉnh thấp hơn
        deep = amp > 0.25 * rng and amp >= min_amp
        valid_frac = float(valid[start:end + 1].mean()) if end > start else 0.0
        reps.append(dict(start=int(start), bottom=int(b), end=int(end), dur=round(dur, 2),
                         valid_frac=round(valid_frac, 2), amp=round(amp, 3),
                         ok=bool(min_rep_s <= dur <= max_rep_s and deep and valid_frac >= min_valid)))
    return reps

## 3. Cấu hình và hàm tạo mẫu (giống hệt notebook 03)

12 khớp, mỗi rep đưa về 32 khung, chuẩn hoá theo từng rep.

In [ ]:
T = 32
JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
JOINT_NAMES = ["L_shoulder", "R_shoulder", "L_elbow", "R_elbow", "L_wrist", "R_wrist",
               "L_hip", "R_hip", "L_knee", "R_knee", "L_ankle", "R_ankle"]
# Cạnh của đồ thị khung xương (chỉ số theo JOINTS), dùng cho ST-GCN và để vẽ
EDGES = [(0, 1), (0, 2), (2, 4), (1, 3), (3, 5), (0, 6), (1, 7), (6, 7), (6, 8), (8, 10), (7, 9), (9, 11)]
SH, HIP = [0, 1], [6, 7]
VIEW_ID = {"front": 0, "side": 1, "diagonal": 2}
LABEL_ID = {"good": 0, "bad": 1}

In [ ]:
def load_video(file):
    d = np.load(os.path.join(KP_ROOT, "keypoints", file))
    det = d["detected"]
    xy = d["kp2d"][:, JOINTS, :2] * np.array([int(d["width"]), int(d["height"])], dtype=np.float32)
    vis = d["kp2d"][:, JOINTS, 3].copy()
    xyz = d["kp3d"][:, JOINTS, :3].copy()

    idx = np.arange(len(det))
    for arr in (xy, xyz):
        flat = arr.reshape(len(det), -1)          # view: sửa flat là sửa arr
        for k in range(flat.shape[1]):
            flat[~det, k] = np.interp(idx[~det], idx[det], flat[det, k])
    vis[~det] = 0.0
    return xy, xyz, vis


def resample(seq, n):
    pos = np.linspace(0, len(seq) - 1, n)
    lo = np.floor(pos).astype(int)
    hi = np.minimum(lo + 1, len(seq) - 1)
    w = (pos - lo).reshape(-1, *[1] * (seq.ndim - 1))
    return seq[lo] * (1 - w) + seq[hi] * w


def normalize_2d(xy):
    mid_sh, mid_hip = xy[:, SH].mean(1), xy[:, HIP].mean(1)
    origin = np.median(mid_hip, axis=0)
    torso = np.median(np.linalg.norm(mid_sh - mid_hip, axis=-1))
    sh_width = np.median(np.linalg.norm(xy[:, SH[0]] - xy[:, SH[1]], axis=-1))
    out = (xy - origin) / (max(torso, sh_width) + 1e-6)
    return np.clip(out, -3, 3)      # chặn các khớp bị MediaPipe đoán văng ra xa (99% giá trị nằm trong ±1.8)


def make_sample(xy, xyz, vis, start, end):
    sl = slice(start, end + 1)
    v = resample(vis[sl], T)[..., None]
    x2d = np.concatenate([resample(normalize_2d(xy[sl]), T), v], axis=-1)     # (T, 12, 3): x, y, visibility
    x3d = np.concatenate([resample(xyz[sl], T), v], axis=-1)                  # (T, 12, 4): x, y, z, visibility
    return x2d.astype(np.float32), x3d.astype(np.float32)

## 4. Đếm rep, cắt rep cho từng clip

Mỗi clip dài khoảng 5 giây nên thường chứa 1–4 rep. Clip mà người tập đứng yên, ra khỏi khung hình
hoặc không thấy rõ tay sẽ không cho ra rep nào.

In [ ]:
X2d, X3d, rows, clip_reps = [], [], [], []
for c in clips:
    file = c["video"].replace(".mp4", ".npz")
    d = np.load(os.path.join(KP_ROOT, "keypoints", file))
    if d["detected"].sum() < 10:
        clip_reps.append(0)
        continue
    sig, valid = rep_signal(d, "front")
    reps = [r for r in find_reps(sig, valid, float(d["fps"]), MIN_AMP["front"]) if r["ok"]]
    clip_reps.append(len(reps))
    if not reps:
        continue
    xy, xyz, vis = load_video(file)
    for k, r in enumerate(reps):
        a, b = make_sample(xy, xyz, vis, r["start"], r["end"])
        X2d.append(a)
        X3d.append(b)
        rows.append(dict(video=c["video"], rep_id=k, worker=c["worker_id"], dur=r["dur"], amp=r["amp"],
                         labels="|".join(c["attrs"])))

X2d, X3d = np.stack(X2d), np.stack(X3d)
meta = pd.DataFrame(rows)
print("Số rep:", len(meta), "| từ", int((np.array(clip_reps) > 0).sum()), "clip | số người:", meta.worker.nunique())
print("Số rep mỗi clip:", pd.Series(clip_reps).value_counts().sort_index().to_dict())
print("X2d:", X2d.shape, "| có NaN:", bool(np.isnan(X2d).any()))

## 5. Quy nhãn QEVD về các loại lỗi của đề tài

| Nhãn QEVD | Quy về |
|---|---|
| `90 degrees` (xuống đúng độ sâu), không kèm lỗi nào | **đúng** |
| `shallow`, `insufficient` | `nong` (xuống không đủ sâu) |
| `elevated hips`, `butt sinking` | `hong` (mông nhô, hông võng) |
| `legs on the floor` | `sap` (chân chạm sàn) |
| `wide`, `very wide`, `shoulder-width`, `narrow` | vị trí tay: là biến thể bài tập, **không tính là lỗi** |

Rep bị loại khỏi tập train/test (`use = False`):
- không phải rep chuẩn hoặc nhãn mơ hồ: `waiting position`, `child pose cheat`, `early stopping`, `not visible`,
  `head moving in an out of frame`, `sloppy`;
- chỉ có nhãn `deep` hoặc `side to side` (QEVD coi là lỗi, nhưng không thuộc ba loại lỗi của đề tài);
- chỉ có nhãn vị trí tay (không biết độ sâu đúng hay sai).

Đầu ra `Y` có 4 cột: `[sai, nong, hong, sap]`.

In [ ]:
ATTRS = ["90 degrees", "shallow", "deep", "elevated hips", "butt sinking", "legs on the floor", "side to side",
         "wide", "very wide", "shoulder-width", "narrow", "fast", "insufficient", "sloppy", "moving head", "head down",
         "waiting position", "child pose cheat", "early stopping", "not visible", "head moving in an out of frame"]
A = np.array([[a in s.split("|") for a in ATTRS] for s in meta["labels"]])
has = {a: A[:, i] for i, a in enumerate(ATTRS)}

drop = has["waiting position"] | has["child pose cheat"] | has["early stopping"] | has["not visible"] \
     | has["head moving in an out of frame"] | has["sloppy"]
nong = has["shallow"] | has["insufficient"]
hong = has["elevated hips"] | has["butt sinking"]
sap = has["legs on the floor"]
bad = nong | hong | sap
good = has["90 degrees"] & ~bad & ~has["side to side"]

use = ~drop & (bad | good)
Y = np.stack([bad, nong, hong, sap], axis=1).astype(np.float32)
ERR_NAMES = ["sai", "nong", "hong", "sap"]

print("Rep dùng được:", int(use.sum()), "| số người:", meta.worker[use].nunique())
print("  đúng:", int(good[use].sum()), "| sai:", int(bad[use].sum()),
      "| nong:", int(nong[use].sum()), "| hong:", int(hong[use].sum()), "| sap:", int(sap[use].sum()))

## 6. Kiểm tra: nhãn có khớp với chuyển động thật không?

`amp` là biên độ của rep: vai hạ xuống bao nhiêu, tính theo độ rộng vai.
Nếu nhãn đáng tin thì phải thấy `insufficient` < `shallow` < `90 degrees` < `deep`.

In [ ]:
table = {}
for a in ["insufficient", "shallow", "90 degrees", "deep", "elevated hips", "butt sinking", "legs on the floor"]:
    v = meta.amp[has[a] & ~drop]
    table[a] = {"số rep": len(v), "p25": v.quantile(0.25), "trung vị": v.median(), "p75": v.quantile(0.75)}
display(pd.DataFrame(table).T.round(2))

## 7. Chia fold theo người tập

5 fold, mỗi người chỉ nằm trong một fold (`GroupKFold`). Số rep mỗi người ít (trung vị khoảng 5) nên các fold khá đều.

In [ ]:
from sklearn.model_selection import GroupKFold

worker = meta.worker.values.astype(np.int64)
fold = np.full(len(meta), -1, dtype=np.int64)
idx = np.where(use)[0]
for k, (_, test) in enumerate(GroupKFold(n_splits=5).split(idx, groups=worker[idx])):
    fold[idx[test]] = k

summary = pd.DataFrame({"fold": fold[use], "worker": worker[use], "sai": Y[use, 0]})
display(summary.groupby("fold").agg(so_nguoi=("worker", "nunique"), so_rep=("sai", "size"), ti_le_sai=("sai", "mean")).round(2))

## 8. Lưu

In [ ]:
np.savez_compressed(
    os.path.join(OUT_DIR, "qevd_dataset.npz"),
    X2d=X2d, X3d=X3d, Y=Y, use=use, fold=fold, worker=worker,
    A=A, attr_names=np.array(ATTRS), err_names=np.array(ERR_NAMES),
    video=np.array(meta.video.tolist()), rep_id=meta.rep_id.values,
    amp=meta.amp.values.astype(np.float32), dur=meta.dur.values.astype(np.float32),
)
meta.assign(use=use, fold=fold, sai=Y[:, 0], nong=Y[:, 1], hong=Y[:, 2], sap=Y[:, 3]) \
    .to_csv(os.path.join(OUT_DIR, "qevd_dataset_meta.csv"), index=False)

## 9. Khớp nào nhìn thấy được ở góc trước?

Trung bình visibility của từng khớp trên các rep dùng được. Gối và cổ chân gần như không thấy vì bị thân che,
nên lỗi về hông (`hong`) rất khó nhận ra ở góc này.

In [ ]:
vis = X2d[use][:, :, :, 2].mean(axis=(0, 1))
display(pd.DataFrame({"visibility": vis}, index=JOINT_NAMES).T.round(2))

## 10. Xem thử vài rep

Mỗi hàng là một rep ở 6 thời điểm. Xương màu đỏ: visibility thấp.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
groups = [("đúng", good & use), ("nong", nong & ~hong & ~sap & use), ("hong", hong & use), ("sap", sap & use)]
times = np.linspace(0, T - 1, 6).astype(int)
fig, axes = plt.subplots(len(groups), 6, figsize=(18, 3 * len(groups)))
for row, (name, mask) in enumerate(groups):
    i = rng.choice(np.where(mask)[0])
    lim = np.abs(X2d[i, :, :6, :2]).max() * 1.2
    for col, t in enumerate(times):
        ax = axes[row, col]
        p, v = X2d[i, t, :, :2], X2d[i, t, :, 2]
        for a, b in EDGES:
            ax.plot([p[a, 0], p[b, 0]], [p[a, 1], p[b, 1]], lw=2, color="tab:green" if min(v[a], v[b]) > 0.5 else "tab:red")
        ax.set_xlim(-lim, lim); ax.set_ylim(lim, -lim); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(f"{name}\n{meta.video[i]}")
        ax.set_title(f"t = {t}")
plt.tight_layout()
plt.show()

## 11. Kết quả

Tải `qevd_dataset.npz` về và tạo Kaggle Dataset **Private** từ nó (hoặc Save Version rồi Add Input notebook này ở bước 7).

In [ ]:
!ls -lh "{OUT_DIR}"